In [6]:
# Copio y pego de a1_derivacion
import pandas as pd
import numpy as np

fuente = "../dataset/airfoil_self_noise.dat"
columnas = [
    "frequency",
    "angle",
    "chord_length",
    "velocity",
    "displacement",
    "sound_level"
]

df = pd.read_csv(
    fuente,
    sep="\t",
    header=None,
    names=columnas
)

print(df)

      frequency  angle  chord_length  velocity  displacement  sound_level
0           800    0.0        0.3048      71.3      0.002663      126.201
1          1000    0.0        0.3048      71.3      0.002663      125.201
2          1250    0.0        0.3048      71.3      0.002663      125.951
3          1600    0.0        0.3048      71.3      0.002663      127.591
4          2000    0.0        0.3048      71.3      0.002663      127.461
...         ...    ...           ...       ...           ...          ...
1498       2500   15.6        0.1016      39.6      0.052849      110.264
1499       3150   15.6        0.1016      39.6      0.052849      109.254
1500       4000   15.6        0.1016      39.6      0.052849      106.604
1501       5000   15.6        0.1016      39.6      0.052849      106.224
1502       6300   15.6        0.1016      39.6      0.052849      104.204

[1503 rows x 6 columns]


In [7]:
import statsmodels.api as sm
import scipy.stats as stats
# Vamos a utilizar esta nueva librería para crear el modelo de regresión lineal, porque nos permite obtener
# de manera más sencilla los valores a completar en el cuadro

variables_del_cuadro = [
    "frequency",
    "angle",
    "chord_length",
    "displacement"
]

y = df["sound_level"].to_numpy()

n = len(y)

for variable in variables_del_cuadro:
    
    x = df[variable]
    
    X = sm.add_constant(x)
    
    modelo = sm.OLS(y, X).fit() # Aca se realiza el ajuste de regresión lineal, para estimar b0 y b1
    
    beta_0 = modelo.params["const"]
    beta_1 = modelo.params[variable]
    
    print(f"Primera columna para {variable}")
    print(f"La recta de regresión lineal de sound_level en función de{variable} es:")
    print(f"y = ({beta_1})x + {beta_0}")
    
    print("*"*10)
    
    print(f"Segunda columna para {variable}")
    sigma2 = modelo.mse_resid
    print(f"sigma² = {sigma2}")
    
    print("*"*10)
    
    print(f"Tercera columna para {variable}")
    
    R2 = modelo.rsquared
    
    print(f"R2 = {R2}")
    
    print("*"*10)
    
    print(f"Cuarta columna para {variable}")
    r = np.corrcoef(x, y)[0,1]
    
    print(f"r = {r}")
    
    print("*"*10)
    
    print(f"Quinta columna para {variable}")
    IC = modelo.conf_int(alpha=0.05)
    IC_beta_0 = tuple(IC.loc["const"])
    
    print(f"IC(Beta0) = {IC_beta_0}")
    
    print("*"*10)

    print(f"Sexta columna para {variable}")
    IC_beta_1 = tuple(IC.loc[variable])
    
    print(f"IC(Beta1) = {IC_beta_1}")
    
    print("+"*30)
    
    # .....................................................
    
    print(f"OPCION 1 texto en función de $x$ : Séptima columna para {variable}")

    x_bar = x.mean()
    Sxx = np.sum((x - x_bar) ** 2)
    se = np.sqrt(modelo.mse_resid)
    t_val = stats.t.ppf(0.975, df=n - 2)  # Valor crítico t al 95%

    # Construcción de la cadena de texto de la fórmula
    margen_base = t_val * se
    ic_expresion = f"({beta_0:.4f} + {beta_1:.4f}*x) ± {margen_base:.4f} * sqrt(1/{n} + (x - {x_bar:.4f})² / {Sxx:.4f})"

    print(f"IC_Respuesta_Media(x) = {ic_expresion}")
    print("*" * 10)
    
    
    # .....................................................
    
    # --- Agregar esto dentro del bucle for ---
    print(f"OPCION 2 ($x^* = x⁻): Séptima columna para {variable}")

    # Usamos el método nativo get_prediction de statsmodels evaluado en [1, x.mean()]
    prediccion = modelo.get_prediction([1, x.mean()])
    IC_resp_media = tuple(prediccion.conf_int(alpha=0.05)[0])

    print(f"IC_Respuesta_Media (en x̄ = {x.mean():.4f}) = {IC_resp_media}")
    print("*" * 10)
    
    print("+"*30)
    
    # .....................................................
    
    
    # --- Agregar esto dentro del bucle for ---
    print(f"Opción 1.texto en función de $x$ Octava columna para {variable}")

    x_bar = x.mean()
    Sxx = np.sum((x - x_bar) ** 2)
    se = np.sqrt(modelo.mse_resid)
    t_val = stats.t.ppf(0.975, df=n - 2)

    margen_base = t_val * se
    ip_expresion = f"({beta_0:.4f} + {beta_1:.4f}*x) ± {margen_base:.4f} * sqrt(1 + 1/{n} + (x - {x_bar:.4f})² / {Sxx:.4f})"

    print(f"IP(x) = {ip_expresion}")
    print("*" * 10)
    
    # .....................................................
    
    # --- Agregar esto dentro del bucle for ---
    print(f"Opcion 2. ($x^* = x⁻$) Octava columna para {variable}")

    prediccion = modelo.get_prediction([1, x.mean()])
    IP_prediccion = tuple(prediccion.conf_int(obs=True, alpha=0.05)[0])

    print(f"IP (en x̄ = {x.mean():.4f}) = {IP_prediccion}")
    print("*" * 10)
     
    

    
    

Primera columna para frequency
La recta de regresión lineal de sound_level en función defrequency es:
y = (-0.0008549790125253054)x + 127.30373759248681
**********
Segunda columna para frequency
sigma² = 40.35323532492145
**********
Tercera columna para frequency
R2 = 0.1526554072390256
**********
Cuarta columna para frequency
r = -0.3907114117082142
**********
Quinta columna para frequency
IC(Beta0) = (126.86789760685716, 127.73957757811647)
**********
Sexta columna para frequency
IC(Beta1) = (-0.0009569642880496154, -0.0007529937370009954)
++++++++++++++++++++++++++++++
OPCION 1 texto en función de $x$ : Séptima columna para frequency
IC_Respuesta_Media(x) = (127.3037 + -0.0009*x) ± 12.4606 * sqrt(1/1503 + (x - 2886.3806)² / 14927953510.3127)
**********
OPCION 2 ($x^* = x⁻): Séptima columna para frequency
IC_Respuesta_Media (en x̄ = 2886.3806) = (np.float64(124.51453371701788), np.float64(125.15735184519107))
**********
++++++++++++++++++++++++++++++
Opción 1.texto en función de $x$ 

In [8]:
import scipy.stats as stats

# --- Agregar esto dentro del bucle for ---
